<a href="https://colab.research.google.com/github/mobadara/precision-diagnostics-xai/blob/main/notebooks/01_model_training_and_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Model Training and Fine-Tuning**

**Objective:** Ingest the Kaggle Chest X-Ray dataset, apply strategic data augmentation, handle severe class imbalance via loss weighting, and fine-tune a pre-trained DenseNet121 architecture to detect pneumonia.

## **Environment Setup**
First, we import the necessary PyTorch libraries and configure our hardware accelerator. Because image processing is computationally expensive, we must ensure PyTorch detects and utilizes the Colab GPU (CUDA).

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os
import numpy as np

# Configure the device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Compute Device: {device}')
if device.type == 'cuda':
    print(f'GPU Model: {torch.cuda.get_device_name(0)}')

Compute Device: cuda
GPU Model: Tesla T4


## **Data Preprocessing and Augmentation Pipeline**
As determined in our EDA, we must standardize all input resolutions to 224x224.

To prevent overfitting without introducing artificial artifacts, we apply light data augmentation **only to the training set** (rotation and color jitter). The validation and test sets remain strictly unaltered to provide an honest evaluation metric.

In [3]:
# Define the dataset paths
data_dir = '../dataset/chest_xray'
train_dir = os.path.join(data_dir, 'train')
val_dir = os.path.join(data_dir, 'val')
test_dir = os.path.join(data_dir, 'test')

# 1. Training Transforms (with Augmentation)
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10), # Slight rotation to simulate patient misalignment
    transforms.ColorJitter(brightness=0.2, contrast=0.2), # Account for different X-ray exposures
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Standard ImageNet normalization
])

# 2. Validation/Testing Transforms (Strictly unaltered)
eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Transforms configured successfully.")

Transforms configured successfully.
